# Introduction

This is a series of notebooks that should be visit in order, they are all linked in the table of content. In this notebook we are going to encode the categorical variables of the transformed data (from previous notebook). At the end we will run a simple XGBoost model to set a performance baseline. Before going on with this notebook, make sure you have finished these courses:

- [Python](https://www.kaggle.com/learn/python)
- [Pandas](https://www.kaggle.com/learn/pandas)
- [Intermediate machine learning](https://www.kaggle.com/learn/intermediate-machine-learning)

#### Content table
- [Preprocessing 1: Data Transformation](https://www.kaggle.com/ponybiam/preprocessing-1-data-transformation)
- **Preprocessing pt. 2: encoding categorical variables** (you are here)
    - [Load and transform data](#Load-and-transform-data)
    - [Encoding](#Encoding)
      - [Identifying categorical features](#Identifying-categorical-features)
      - [Integer encoding](#Integer-encoding)
      - [One hot encoding](#One-hot-encoding)
    - [Model](#Model)
- [Preprocessing 3: Handling Missing Values](https://www.kaggle.com/ponybiam/preprocessing-3-handling-missing-values) 
- [Feature engineering 1: Simple Features](https://www.kaggle.com/ponybiam/feature-engineering-1-simple-features)
- [Feature engineering 2: Clustering & PCA](https://www.kaggle.com/ponybiam/feature-engineering-2-clustering-pca)
- [Feature engineering 3: Target Encoding](https://www.kaggle.com/ponybiam/feature-engineering-3-target-encoding)

In [ ]:
import pandas as pd
import numpy as np

# Notebook goal
The goal of this notebook is encode  the categorical features and run a simple predictive model. Here will be shown only one of the many possibilities in machine learning, feel free to experiment and play around with the data on your own. We will be focusing on the techniques and not on the model performance.

# Load and transform data
In our [previous notebook](https://www.kaggle.com/ponybiam/preprocessing-pt-1-data-transformation) we transformed our data to be used in a predictive model. At the end of it we wrote a function to perform all the process explained there, at once. Don't worry about this function, just make sure you checked and understood the transformation in the previous notebook.

In [ ]:
def transform_dataset(metadata_path, weather_path, energy_path, site_id):
    """
    metadata_path: path to metadata data set
    weather_path: path to weather data set
    energy_path: path to energy data set
    site_id: selected site id
    """
    # Load data
    metadata = pd.read_csv(metadata_path)
    weather = pd.read_csv(weather_path, parse_dates=["timestamp"])
    energy = pd.read_csv(energy_path, parse_dates=["timestamp"])
    
    # Filter
    metadata = metadata.loc[metadata.site_id == site_id, ["building_id", "site_id", "primaryspaceusage", "sqm"]]
    weather = weather.loc[weather.site_id == site_id]
    cols = ["timestamp"] + [col for col in energy.columns if site_id in col]
    energy = energy[cols]
    
    # Melt
    energy = energy.melt(id_vars="timestamp", var_name="building_id", value_name="meter_reading")
    
    # Merge
    energy = pd.merge(energy, metadata, how="left", on="building_id")
    energy = pd.merge(energy, weather, how="left", on=["timestamp","site_id"])
    
    return energy

Remember, we are working with **electricity** data and only the site **Panther**.

In [ ]:
metadata_path = "/kaggle/input/buildingdatagenomeproject2/metadata.csv"
weather_path = "/kaggle/input/buildingdatagenomeproject2/weather.csv"
energy_path = "/kaggle/input/buildingdatagenomeproject2/electricity.csv"
site_id = "Panther"

df = transform_dataset(metadata_path, weather_path, energy_path, site_id)
df.head()

In [ ]:
df.info()

# Encoding

## Identifying categorical features
Categorical features are usually of type `object` (i.e., an string). Let's inspect our data set:

In [ ]:
df.info()

And you can get the names of the `object` columns so you don't have to write them down:

In [ ]:
# Get list of categorical variables
s = (df.dtypes == 'object')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

Clearly, there are two kind of features here:
- **Identification features:** `building_id` and `site_id`. They assign a unique name to each building and site; in this case we have objects, but these could be integer numbers and would work exactly the same.
- **Information features:** `primaryspaceusage`. This feature give us information about each row of our data, it tell us to which usage category they belong.

## Integer encoding
*Integer encoding* or *Ordinal encoding* is a simple technique that consist of assigning an integer number to each category of our column. For example, a person could have features `["male", "female"]`, `["from Europe", "from US", "from Asia"]`, `["uses Firefox", "uses Chrome", "uses Safari", "uses Internet Explorer"]`. Such features can be efficiently coded as integers, for instance `["male", "from US", "uses Internet Explorer"]` could be expressed as `[0, 1, 3]` while `["female", "from Asia", "uses Chrome"]` would be `[1, 2, 1]`. This approach also works when there is an ordering of the categories: `["Never", "Rarely", "Most days", "Every day"]` could be encoded as `[0, 1, 2, 3]`.

In our data set, this approach would work for our identification features: `building_id` and `site_id`. For us (humans) is almost the same to identify each bulding/site with a name or number, but predictive models need numerical features to work. We are going to use [OrdinalEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) from *ScikitLearn* package.

**Remember**: here we are working with only one site, so there is only one category in `site_id`. We are going to encode it anyway, but have in mind that in this particular case is not usefull information.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

First, define the encoder and the columns you want to encode:

In [ ]:
# Define the encoder
encoder = OrdinalEncoder()

# Columns to encode
cols = ["building_id","site_id"]

Now is time fot the encoding. We are using the method [*fit_transform*](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder.fit_transform), this will fit the model (i.e., assign a different number to each of the categories present in the dataframe we pass) and return and encoded dataframe.

In [ ]:
# This returns an array
encoded_columns = encoder.fit_transform(df[cols])

# Check it out
encoded_columns

This method returns a plain array. We can convert it to a dataframe:

In [ ]:
# We can convert it to a dataframe
encoded_columns = pd.DataFrame(encoded_columns)

# Check it out
encoded_columns.head()

But the column names are gone. We can easily assign them again, remember we have them in our `cols` variable:

In [ ]:
# And we asign the column names (the encoder remove them)
encoded_columns.columns = cols

# Check it out
encoded_columns.head()

And finally, we repace it in our dataset:

In [ ]:
# And replace it in your data
df[cols] = encoded_columns

df.head()

If you want your original categories you can use `inverse_trasnsform`:

In [ ]:
encoder.inverse_transform(df[cols])

## One hot encoding
In cases when there is no ordinal relationship between the categories, using ordinal encoding can mislead our predictive model: the model will asume a natural order in our categories. For this cases is better to use [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html): this technique create a binary column per category, indicating with 0 or 1 to which category the row belongs.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

### Primary space usage
First we have to inspect our feature to check how many categories we have.

In [ ]:
print("List of categories:")
print(list(df.primaryspaceusage.unique()))
print("Total categories:")
print(len(df.primaryspaceusage.unique()))

7 categories is not a lot, but considering we are creating one column per category, let's try out to reduce that number. Maybe we could group some minor categories?

In [ ]:
# First we count how many rows are in each category
count = df.primaryspaceusage.value_counts()

# And we convert it o percentage
count / len(df) * 100

So we have 3 categories that have around 75% of the rows. We could group all the other cotegories together, let's try it out.

In [ ]:
# Get columns with less than 10% of data
s = (df.primaryspaceusage.value_counts() < len(df)*0.1)

# Convert to list
categories_under_10pct = list(s[s].index)
print(categories_under_10pct)

In [ ]:
# Replace for category "Other"
df = df.replace(categories_under_10pct, "Other")

# Check
df.primaryspaceusage.value_counts() / len(df) * 100

### Encoding
Now that we have less categories, is time to encode them. The process is similar to the one we followed for the OrdinalEncoder.

In [ ]:
# First, create the encoder
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

# Choose the columns to encode
cols = ["primaryspaceusage"]

# Encode the columns
OH_cols = OH_encoder.fit_transform(df[cols])

# Convert it to a dataframe
OH_cols = pd.DataFrame(OH_cols)

# And we can rename the columns instead of using just numbers
OH_cols.columns = ["primary_use_0", "primary_use_1", "primary_use_2", "primary_use_3"]

# Check it out
OH_cols.head()

And now is time to replace them in our dataset

In [ ]:
# Remove categorical columns (will be replaced with one-hot encoding)
df = df.drop(cols, axis=1)

# Add one-hot encoded columns to numerical features
df = pd.concat([df, OH_cols], axis=1)

In [ ]:
df.head()

And if you want the original values, you can decode them with `inverse_transform`.

In [ ]:
OH_encoder.inverse_transform(df[["primary_use_0", "primary_use_1", "primary_use_2", "primary_use_3"]])

# Model

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error



We haven't yet handle missing values, so, for now, we are **replacing them with 0**. Which is a terrible practice, but we'll learn about it later.

In [ ]:
# Replace the missing values
df = df.replace(np.nan, 0)

We are working with a two years data. Let's split it like this:
- **Training**: first year of data
- **Validation**: part of the second year of data

Pay attention to this, with the following line we are:
- filtering the rows based on the date, we can do that because `timestamp` is a datetime object, that's what's the `parse_dates` in `pandas.read_csv` does
- setting `timestamp` as the index

In [ ]:
# Train set
X_train = df[df.timestamp < "2017-01-01"].set_index("timestamp").drop("meter_reading", axis=1)
y_train = df[df.timestamp < "2017-01-01"].set_index("timestamp").meter_reading

# Validation set
X_val = df[df.timestamp >= "2017-07-01"].set_index("timestamp").drop("meter_reading", axis=1)
y_val =df[df.timestamp >= "2017-07-01"].set_index("timestamp").meter_reading

And this is the metric we are going to use:

In [ ]:
def RMSLE(y_true, y_pred):
    """
    The Root Mean Squared Log Error (RMSLE) metric 

    :param y_true: The ground truth labels given in the dataset
    :param y_pred: Our predictions
    :return: The RMSLE score
    """
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

Time to train our model!

In [ ]:
# This is the model we are going to train (a simple one)
my_model = XGBRegressor(n_estimators=50, learning_rate=0.1, n_jobs=4, random_state=55, objective="reg:squaredlogerror")

# Train
print("Training...")
my_model.fit(X_train, y_train, verbose=False)
print("...done")

In [ ]:
# And we predict
prediction = pd.DataFrame({"y_pred": my_model.predict(X_val)})

In [ ]:
# In case there is a negative or infinite prediction (could happens) we replace them with 0
prediction.replace([np.inf, -np.inf], np.nan, inplace=True) # replace inf
prediction[prediction <0] = np.nan # replace negative values
prediction.fillna(0, inplace=True) # replace all missings with 0

# Calculate the metric
rmsle = RMSLE(y_val, prediction.y_pred)
rmsle

# (Optional) A function to do it all
This part is optional. We are going to write functions that follows all the steps we performed here. These functions will be used in the following notebooks.

In [ ]:
def encode_categorical(df):
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.preprocessing import OneHotEncoder
    
    ordinal_cols = ["building_id","site_id"]
    cols = ["primaryspaceusage"] # only one here, but you could add more
    
    # Ordinal encoder
    # Define the encoder
    encoder = OrdinalEncoder()

    # This returns an array
    encoded_columns = encoder.fit_transform(df[ordinal_cols])

    # We can convert it to a dataframe
    encoded_columns = pd.DataFrame(encoded_columns)

    # And we asign the column names (the encoder remove them)
    encoded_columns.columns = ordinal_cols

    # And replace it in your data
    df[ordinal_cols] = encoded_columns
    
    # One Hot Encoder
    for col in cols:
        # Get columns with less than 10% of data
        s = (df[col].value_counts() < len(df)*0.1)
        # Convert to list
        categories_under_10pct = list(s[s].index)
        # Replace for category "Other"
        df = df.replace(categories_under_10pct, "Other")
        
    OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    
    # Transform
    OH_cols = pd.DataFrame(OH_encoder.fit_transform(df[cols]))
    
    # Rename
    OH_cols.columns = ["primary_use_0", "primary_use_1", "primary_use_2", "primary_use_3"]
    
    # Remove categorical columns (will be replaced with one-hot encoding)
    df = df.drop(cols, axis=1)

    # Add one-hot encoded columns to numerical features
    df = pd.concat([df, OH_cols], axis=1)
    
    return df

In [ ]:
def train_model(df):
    # Imports
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_squared_log_error
    
    # Metric
    def RMSLE(y_true, y_pred):
        return np.sqrt(mean_squared_log_error(y_true, y_pred))
    
    # Split data
    # Train set
    X_train = df[df.timestamp < "2017-01-01"].set_index("timestamp").drop("meter_reading", axis=1)
    y_train = df[df.timestamp < "2017-01-01"].set_index("timestamp").meter_reading
    # Validation set
    X_val = df[df.timestamp >= "2017-07-01"].set_index("timestamp").drop("meter_reading", axis=1)
    y_val =df[df.timestamp >= "2017-07-01"].set_index("timestamp").meter_reading
    
    # Train
    # This is the model we are going to train (a simple one)
    my_model = XGBRegressor(n_estimators=50, learning_rate=0.1, n_jobs=4, random_state=55, objective="reg:squaredlogerror")
    # Train
    my_model.fit(X_train, y_train, verbose=False)
    
    # Predict
    prediction = pd.DataFrame({"y_pred": my_model.predict(X_val)})
    
    # Process prediction
    # In case there is a negative or infinite prediction (could happens) we replace them with 0
    prediction.replace([np.inf, -np.inf], np.nan, inplace=True) # replace inf
    prediction[prediction <0] = np.nan # replace negative values
    prediction.fillna(0, inplace=True) # replace all missings with 0

    # Calculate the metric
    rmsle = RMSLE(y_val, prediction.y_pred)
    
    return rmsle